In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [2]:
spark = (
    SparkSession.builder\
    .appName("Office")\
    .config("spark.driver.memory", "5g")\
    .config("spark.executor.memory", "5g")\
    .config("spark.driver.maxResultSize", "2g")\
    .config("spark.sql.shuffle.partitions", "200")\
    .config("spark.sql.caseSensitive", "true")\
    .getOrCreate()
)

your 131072x1 screen size is bogus. expect trouble
26/03/09 00:35:35 WARN Utils: Your hostname, kien resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/09 00:35:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 00:35:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df_office = spark.read.format("json").option("header", "true").option("inferSchema", "true").load("../Office_Products.jsonl")

In [4]:
# Chọn các cột cần thiết, bao gồm parent_asin để join với metadata
df_office = df_office.select(["user_id", "asin", "parent_asin", "rating", "timestamp", "verified_purchase"])

In [5]:
df_office = df_office.withColumn("ts",from_unixtime(col("timestamp") / 1000))\
    .withColumn("date", to_date("ts"))\
    .withColumn("year", year("ts"))\
    .withColumn("month", month("ts"))\
    .withColumn("day", dayofmonth("ts"))


In [6]:
df_office.show(5, truncate=False)

+----------------------------+----------+-----------+------+-------------+-----------------+-------------------+----------+----+-----+---+
|user_id                     |asin      |parent_asin|rating|timestamp    |verified_purchase|ts                 |date      |year|month|day|
+----------------------------+----------+-----------+------+-------------+-----------------+-------------------+----------+----+-----+---+
|AFKZENTNBQ7A7V7UXW5JJI6UGRYQ|B01AHHL4X2|B01MZ3SD2X |5.0   |1677939345945|true             |2023-03-04 21:15:45|2023-03-04|2023|3    |4  |
|AFKZENTNBQ7A7V7UXW5JJI6UGRYQ|B08L6H23JZ|B08L6H23JZ |4.0   |1677939160682|true             |2023-03-04 21:12:40|2023-03-04|2023|3    |4  |
|AFKZENTNBQ7A7V7UXW5JJI6UGRYQ|B07JDZ5J46|B07JDZ5J46 |1.0   |1660188831933|true             |2022-08-11 10:33:51|2022-08-11|2022|8    |11 |
|AFKZENTNBQ7A7V7UXW5JJI6UGRYQ|B004MNX7EW|B07BR2PBJN |4.0   |1659806066713|true             |2022-08-07 00:14:26|2022-08-07|2022|8    |7  |
|AFKZENTNBQ7A7V7UXW5JJI6UGR

In [7]:
meta_schema = StructType([
    StructField("main_category", StringType(), True),
    StructField("title", StringType(), True),
    StructField("average_rating", DoubleType(), True),
    StructField("rating_number", LongType(), True),
    StructField("features", ArrayType(StringType()), True),
    StructField("description", ArrayType(StringType()), True),
    StructField("price", DoubleType(), True),
    StructField("images", ArrayType(MapType(StringType(), StringType())), True),
    StructField("videos", ArrayType(MapType(StringType(), StringType())), True),
    StructField("store", StringType(), True),
    StructField("categories", ArrayType(StringType()), True),

    StructField("details", StringType(), True),

    StructField("parent_asin", StringType(), True),
    StructField("bought_together", DoubleType(), True),
    StructField("subtitle", StringType(), True),
    StructField("author", StringType(), True),
])


In [8]:
df_meta = spark.read.format("json").option("header", "true").schema(meta_schema).load("../meta_Office_Products.jsonl")

In [9]:
df_meta.show(5, truncate=False)

+---------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [10]:
df_meta = df_meta.select("title", "store", "categories", "parent_asin")

In [11]:
df_meta.show(5)

+--------------------+-----------------+--------------------+-----------+
|               title|            store|          categories|parent_asin|
+--------------------+-----------------+--------------------+-----------+
|Alliance Rubber 0...|         Alliance|[Office Products,...| B001S28Q4Q|
|10 Set Mini Offic...|           Chneeu|[Office Products,...| B07RPQVH21|
|Creanoso Humorous...|         Creanoso|[Office Products,...| B07J58B87Q|
|PenPower WorldPen...|         PenPower|[Office Products,...| B098NQ685V|
|Assorted Best Sel...|Apartment 2 Cards|[Office Products,...| B00DBSNIC0|
+--------------------+-----------------+--------------------+-----------+
only showing top 5 rows



In [12]:
meta_count = df_meta.count()
df_office_count = df_office.count()
print (meta_count, df_office_count)

710503 12845712


In [13]:
# Loại bỏ dữ liệu trùng
df_meta = df_meta.dropDuplicates(["parent_asin"])
print(f"Total records after deduplication: {df_meta.count()}")

Total records after deduplication: 710503


In [ ]:
df_office = df_office.dropDuplicates(["user_id", "asin", "ts"])
print(f"Total records after deduplication: {df_office.count()}")

Total records after deduplication: 12715095


In [15]:
# Thay thế dữ liệu cột bị null
df_meta = df_meta.withColumn("store", when(col("store").isNull(), "Unknown").otherwise(col("store")))

In [16]:
# Lọc dữ liệu hợp lệ cho hệ khuyến nghị


# Chỉ giữ rating từ 1-5
df_office_valid = df_office.filter((F.col("rating") >= 1) & (F.col("rating") <= 5))

# Loại bỏ user_id, asin, parent_asin null
df_office_valid = df_office_valid.filter(
    F.col("user_id").isNotNull() & 
    F.col("asin").isNotNull() & 
    F.col("parent_asin").isNotNull()
)

df_office = df_office_valid.cache()
print(f"Reviews hợp lệ (rating 1-5, không null): {df_office.count()}")

Reviews hợp lệ (rating 1-5, không null): 12715094


In [ ]:
# Tạo user-item interaction dataset (chỉ giữ reviews có trong metadata)


# Join trên parent_asin (Spark giữ 1 cột khi dùng tên cột)
df_interactions = df_office.join(df_meta.select("parent_asin"), "parent_asin", "inner")

# Gộp nhiều review cùng user+product (khác variant asin): lấy rating trung bình
df_interactions = df_interactions.groupBy("user_id", "parent_asin").agg(
    F.avg("rating").alias("rating"),
    F.max("ts").alias("ts")
)

print(f"Số lượng tương tác user-item: {df_interactions.count()}")
df_interactions.show(5, truncate=False)      


26/03/09 00:39:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 00:39:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 00:39:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 00:39:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 00:39:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 00:39:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 00:39:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 00:39:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/09 00:39:39 WARN RowBasedKeyValueBatch: Calling spill() on

Số lượng tương tác user-item: 12689348


+----------------------------+-----------+------+-------------+
|user_id                     |parent_asin|rating|timestamp    |
+----------------------------+-----------+------+-------------+
|AFWZYFSXUJEBXLFJSFI3QQL4MOUQ|0008529426 |5.0   |1675220665026|
|AF5PHW7BPQPIKBYHRECPSBPVM2QQ|0008554404 |5.0   |1671132455593|
|AGP5W4XNLLC32Q5ZPAQB6EMK3RTQ|0060515597 |5.0   |1460118763000|
|AFCJPKH344LOVYMIBLUXLUFU2ZZQ|0060515597 |5.0   |1058458591000|
|AEFFH2KEWUM45O3SKRZ647DN2VEA|0060515597 |4.0   |1067582810000|
+----------------------------+-----------+------+-------------+
only showing top 5 rows



In [ ]:
# Lọc cold-start: loại bỏ user/item có quá ít tương tác
MIN_USER_INTERACTIONS = 5   # User phải có >= 5 review
MIN_ITEM_INTERACTIONS = 5   # Sản phẩm phải có >= 5 review



# Lọc iterative: loại user/item ít tương tác, lặp đến khi ổn định
for i in range(5):
    user_counts = df_interactions.groupBy("user_id").count()
    item_counts = df_interactions.groupBy("parent_asin").count()
    
    df_filtered = df_interactions.join(
        user_counts.filter(F.col("count") >= MIN_USER_INTERACTIONS).select("user_id"),
        "user_id", "inner"
    ).join(
        item_counts.filter(F.col("count") >= MIN_ITEM_INTERACTIONS).select("parent_asin"),
        "parent_asin", "inner"
    )
    
    
    df_interactions = df_filtered

   

print(f"Sau lọc cold-start: {df_interactions.count()} tương tác")
print(f"Số users: {df_interactions.select('user_id').distinct().count()}")
print(f"Số items: {df_interactions.select('parent_asin').distinct().count()}")

In [ ]:
# Lưu dữ liệu đã xử lý cho mô hình khuyến nghị
import os
OUTPUT_DIR = "../output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# User-item matrix: user_id, item_id (parent_asin), rating, timestamp
df_recommender = df_interactions.select(
    F.col("user_id").alias("user_id"),
    F.col("parent_asin").alias("item_id"),
    F.col("rating").alias("rating"),
    F.col("ts").alias("ts")
)


df_recommender.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/office_products_interactions.parquet")




In [ ]:
# Lưu metadata sản phẩm (chỉ các sản phẩm có trong interactions)
valid_items = df_interactions.select("parent_asin").distinct()
df_meta_clean = df_meta.join(valid_items, "parent_asin", "inner")
df_meta_clean.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/office_products_metadata.parquet")
